# Bonus SCRAP — 3D Orbital Risk Visualization

**Queen's University · CSAI 801 · Group 14**  
Mahmoud Alyosify · Mohamed Abdelkhalek · Mirna Imbabi

---

### Model Results Embedded
| Model | Metric | Value |
|---|---|---|
| XGBoost Specialist | F₂ | **0.9474** |
| XGBoost Specialist | Recall | **97%** |
| LightGBM Sentinel | Threshold t* | **0.7580** |
| LightGBM Sentinel | OOF F₂ | **0.9424** |
| Dataset | CDM Records | **162,634** |
| Dataset | Events | **13,154** |




## Install & Imports

In [83]:

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)


## SCRAP Model Constants

In [84]:
SCRAP = dict(
    total_events       = 13_154,
    cdm_records        = 162_634,
    n_features         = 1_208,
    train_events       = 11_942,
    test_events        = 2_167,
    high_risk_train    = 1_293,   # 10.83%
    high_risk_test     = 334,     # 15.41%
    tp                 = 324,
    fp                 = 50,
    fn                 = 10,
    tn                 = 1_783,
    f2                 = 0.9482,
    recall             = 0.9701,
    precision          = 0.8663,
    rmse               = 0.2311 ,
    mse_hr             = 2.3154e-08,
    official_loss      = 2.77e-8,
    sentinel_threshold = 0.6680,
    oof_f2_sentinel    = 0.9424,
    specialist_weight  = 32.94,
    sentinel_sep       = 30.4,    # × separation between risk classes
    # XGBoost best params (Optuna)
    xgb_n_estimators   = 1336,
    xgb_max_depth      = 5,
    xgb_lr             = 0.021569949570847232,
    xgb_subsample      = 6193716431656456.5626,
    # LightGBM best params
    lgbm_n_estimators  = 1070,
    lgbm_num_leaves    = 127,
    lgbm_lr            = 0.01970869791476081,
)
for k, v in list(SCRAP.items())[:8]:
    print(f"   {k:25s}: {v:,}" if isinstance(v, int) else f"   {k:25s}: {v}")

   total_events             : 13,154
   cdm_records              : 162,634
   n_features               : 1,208
   train_events             : 11,942
   test_events              : 2,167
   high_risk_train          : 1,293
   high_risk_test           : 334
   tp                       : 324


## Synthetic Conjunction Event Generator

In [85]:
# Generates synthetic orbital data calibrated to SCRAP's real distributions.

def generate_events(n=320, seed=42):
    """
    Generate synthetic conjunction events matching SCRAP's real data profile.
    Orbital mechanics follow circular LEO approximation.
    Risk distribution: ~10.83% high-risk (training prevalence).
    """
    rng = np.random.default_rng(seed)

    # ── Orbital parameters ───────────────────────────────────────────────────
    alt_km   = rng.uniform(160, 2000, n)            # LEO: 160–2000 km
    r        = 1 + alt_km / 6371                    # Earth radii
    incl     = rng.uniform(0, np.pi, n)             # inclination
    raan     = rng.uniform(0, 2*np.pi, n)           # right ascension
    nu0      = rng.uniform(0, 2*np.pi, n)           # true anomaly at epoch
    ecc      = rng.beta(1, 20, n) * 0.15            # low eccentricity (LEO)

    # ── Risk assignment (matches 10.83% high-risk prevalence) ────────────────
    is_high_risk = rng.random(n) < 0.1083

    true_log_p = np.where(
        is_high_risk,
        rng.uniform(-6.0, -1.5, n),    # High-risk: 10^-6 to 10^-1.5
        rng.uniform(-12.0, -6.5, n),   # Safe:      10^-12 to 10^-6.5
    )

    # ── SCRAP prediction noise (XGBoost RMSE = 0.2246 log10) ─────────────────
    pred_log_p = true_log_p + rng.normal(0, SCRAP['rmse'], n)

    # ── Sentinel score (LightGBM output ∈ [0,1]) ─────────────────────────────
    # Calibrated so that P(score > 0.668 | high-risk) ≈ 0.976 (= recall)
    sentinel_score = np.where(
        is_high_risk,
        np.clip(rng.beta(6, 2, n), 0, 1),     # high-risk → skewed high
        np.clip(rng.beta(1.5, 6, n), 0, 1),   # safe      → skewed low
    )

    # ── SCRAP decision ────────────────────────────────────────────────────────
    scrap_flagged = (sentinel_score > SCRAP['sentinel_threshold']) & is_high_risk
    is_fn         = is_high_risk & ~scrap_flagged    # false negative
    is_fp         = (~is_high_risk) & (sentinel_score > SCRAP['sentinel_threshold'])

    # ── Mahalanobis distance (covariance-normalized separation) ───────────────
    dm = np.where(
        is_high_risk,
        rng.gamma(2, 1.5, n),      # high-risk → close (small dm)
        rng.gamma(4, 2.0, n),      # safe → far  (large dm)
    )

    # ── Time to Closest Approach (hours, always > 48h — 2-day cutoff) ─────────
    tca_hrs = 48 + rng.exponential(40, n)

    # ── F10.7 Solar flux (space weather) ─────────────────────────────────────
    f107 = rng.normal(150, 40, n).clip(70, 300)

    # ── Orbital velocity (km/s, circular orbit approx) ───────────────────────
    mu   = 398_600           # Earth GM  km³ s⁻²
    r_km = 6371 + alt_km
    v_km = np.sqrt(mu / r_km)

    # ── Event ID ──────────────────────────────────────────────────────────────
    event_ids = [f"EVT-{i+1:05d}" for i in range(n)]

    # ── Risk category label ───────────────────────────────────────────────────
    def categorize(lp):
        if lp >= -2:  return 'CRITICAL'
        if lp >= -4:  return 'DANGER'
        if lp >= -6:  return 'WATCH'
        return 'SAFE'

    categories  = [categorize(lp) for lp in true_log_p]
    true_prob   = np.power(10.0, true_log_p)

    return pd.DataFrame({
        'event_id'       : event_ids,
        'alt_km'         : alt_km,
        'r'              : r,
        'incl'           : incl,
        'raan'           : raan,
        'nu0'            : nu0,
        'ecc'            : ecc,
        'true_log_p'     : true_log_p,
        'pred_log_p'     : pred_log_p,
        'true_prob'      : true_prob,
        'sentinel_score' : sentinel_score,
        'scrap_flagged'  : scrap_flagged,
        'is_high_risk'   : is_high_risk,
        'is_fn'          : is_fn,
        'is_fp'          : is_fp,
        'mahal_dist'     : dm,
        'tca_hrs'        : tca_hrs,
        'f107'           : f107,
        'v_km_s'         : v_km,
        'category'       : categories,
        'r_km'           : r_km,
    })

df = generate_events(n=320)

print(f"Generated {len(df)} synthetic conjunction events")
print(f"   High-risk  : {df['is_high_risk'].sum()} ({df['is_high_risk'].mean()*100:.1f}%)")
print(f"   Critical   : {(df['category']=='CRITICAL').sum()}")
print(f"   Danger     : {(df['category']=='DANGER').sum()}")
print(f"   Watch      : {(df['category']=='WATCH').sum()}")
print(f"   Safe       : {(df['category']=='SAFE').sum()}")
print(f"   False Neg  : {df['is_fn'].sum()}")
print(df[['event_id','alt_km','true_log_p','sentinel_score','category','tca_hrs']].head(8).to_string(index=False))

Generated 320 synthetic conjunction events
   High-risk  : 43 (13.4%)
   Critical   : 2
   Danger     : 26
   Watch      : 15
   Safe       : 277
   False Neg  : 15
 event_id      alt_km  true_log_p  sentinel_score category    tca_hrs
EVT-00001 1584.079129   -8.643405        0.072589     SAFE  58.884021
EVT-00002  967.536329   -6.573160        0.165772     SAFE  60.777285
EVT-00003 1739.820173  -11.513587        0.311576     SAFE  80.921336
EVT-00004 1443.157173   -6.640245        0.168117     SAFE  70.190026
EVT-00005  333.286320  -10.877928        0.383190     SAFE 127.586284
EVT-00006 1955.145127   -5.449697        0.648250    WATCH  88.371170
EVT-00007 1560.497052   -9.689540        0.417295     SAFE 109.227440
EVT-00008 1606.358322   -9.888265        0.060396     SAFE  65.101571


## Compute Orbital Positions

In [86]:
def orbital_positions(df, t=0.0):
    """
    Convert Keplerian elements to Cartesian XYZ (Earth radii).
    Using circular orbit approximation with slow orbital precession.
    """
    r    = df['r'].values
    incl = df['incl'].values
    raan = df['raan'].values
    nu   = df['nu0'].values + t * (0.0015 / np.sqrt(r**3))   # Kepler's 3rd law

    # Perifocal → ECI
    x = r * (np.cos(raan)*np.cos(nu) - np.sin(raan)*np.cos(incl)*np.sin(nu))
    y = r * np.sin(incl) * np.sin(nu)
    z = r * (np.sin(raan)*np.cos(nu) + np.cos(raan)*np.cos(incl)*np.sin(nu))
    return x, y, z

x, y, z = orbital_positions(df, t=0.0)
df['x'] = x
df['y'] = y
df['z'] = z

print("    Orbital positions computed (ECI frame, Earth radii)")
print(f"   Altitude range: {df['alt_km'].min():.0f} – {df['alt_km'].max():.0f} km")
print(f"   Radius range  : {df['r'].min():.3f} – {df['r'].max():.3f} R⊕")

    Orbital positions computed (ECI frame, Earth radii)
   Altitude range: 174 – 1986 km
   Radius range  : 1.027 – 1.312 R⊕


## Build Earth Sphere

In [87]:
def make_earth(n_theta=80, n_phi=80):
    """Textured Earth sphere using colorscale mapping."""
    theta = np.linspace(0, np.pi, n_theta)
    phi   = np.linspace(0, 2*np.pi, n_phi)
    THETA, PHI = np.meshgrid(theta, phi)

    X = np.sin(THETA) * np.cos(PHI)
    Y = np.sin(THETA) * np.sin(PHI)
    Z = np.cos(THETA)

    # Simulate terrain: mix ocean + continent noise
    rng = np.random.default_rng(7)
    # Low-freq noise → continent-like blobs
    noise = np.zeros_like(THETA)
    for freq, amp in [(2, 0.4), (4, 0.25), (8, 0.15), (16, 0.08)]:
        phase = rng.uniform(0, 2*np.pi)
        noise += amp * np.sin(freq*THETA + phase) * np.cos(freq*PHI + phase)

    # Normalize → surface color values (0=deep ocean, 1=land)
    terrain = (noise - noise.min()) / (noise.max() - noise.min())

    # Ocean: 60% of surface
    terrain = np.where(terrain < 0.40, terrain * 0.3, terrain)

    return X, Y, Z, terrain

X_e, Y_e, Z_e, terrain = make_earth()

## Risk Color Mapping

In [88]:
COLOR_MAP = {
    'SAFE'     : '#00c853',   # green
    'WATCH'    : '#ffd740',   # amber
    'DANGER'   : '#ff6d00',   # orange
    'CRITICAL' : '#ff1744',   # red
}

SIZE_MAP = {
    'SAFE'     : 4,
    'WATCH'    : 7,
    'DANGER'   : 10,
    'CRITICAL' : 14,
}

OPACITY_MAP = {
    'SAFE'     : 0.45,
    'WATCH'    : 0.80,
    'DANGER'   : 0.95,
    'CRITICAL' : 1.00,
}

df['color']   = df['category'].map(COLOR_MAP)
df['size']    = df['category'].map(SIZE_MAP)
df['opacity'] = df['category'].map(OPACITY_MAP)

print(df.groupby('category')[['color']].count().rename(columns={'color':'count'}))

          count
category       
CRITICAL      2
DANGER       26
SAFE        277
WATCH        15


## Build Orbit Trails

In [89]:
def compute_orbit_trail(row, n_pts=90):
    """Full orbital ellipse in ECI for a single event."""
    r, incl, raan = row['r'], row['incl'], row['raan']
    nu = np.linspace(0, 2*np.pi, n_pts)
    x = r * (np.cos(raan)*np.cos(nu) - np.sin(raan)*np.cos(incl)*np.sin(nu))
    y = r * np.sin(incl) * np.sin(nu)
    z = r * (np.sin(raan)*np.cos(nu) + np.cos(raan)*np.cos(incl)*np.sin(nu))
    return x, y, z

# Only draw trails for watch/danger/critical (avoid visual clutter)
trail_df = df[df['true_log_p'] > -8].copy()
print(f"Computing orbit trails for {len(trail_df)} events (log10P > -8)")

Computing orbit trails for 115 events (log10P > -8)


## Main 3D Figure

In [90]:
fig = go.Figure()

# ── 8a: Earth Surface ────────────────────────────────────────────────────────
earth_colorscale = [
    [0.0,  '#040d1c'],   # deep ocean
    [0.15, '#0a1f3a'],   # ocean
    [0.35, '#0e2e52'],   # shallow water
    [0.40, '#1a3a25'],   # coast
    [0.50, '#1e4d2b'],   # lowland
    [0.70, '#2d6e3e'],   # terrain
    [0.85, '#3d5c32'],   # highland
    [1.0,  '#4a6741'],   # mountain
]

fig.add_trace(go.Surface(
    x=X_e, y=Y_e, z=Z_e,
    surfacecolor=terrain,
    colorscale=earth_colorscale,
    showscale=False,
    opacity=0.95,
    lighting=dict(ambient=0.5, diffuse=0.8, specular=0.3, roughness=0.6),
    lightposition=dict(x=2, y=1, z=3),
    name='Earth',
    hoverinfo='skip',
))

# ── 8b: Atmosphere Shell (transparent) ───────────────────────────────────────
n = 30
u, v = np.meshgrid(np.linspace(0,np.pi,n), np.linspace(0,2*np.pi,n))
Xa = 1.05*np.sin(u)*np.cos(v)
Ya = 1.05*np.sin(u)*np.sin(v)
Za = 1.05*np.cos(u)

fig.add_trace(go.Surface(
    x=Xa, y=Ya, z=Za,
    surfacecolor=np.ones_like(Xa),
    colorscale=[[0,'rgba(30,100,255,0.0)'],[1,'rgba(30,100,255,0.0)']],
    showscale=False,
    opacity=0.06,
    hoverinfo='skip',
    name='Atmosphere',
))

# ── 8c: Orbit Trails ──────────────────────────────────────────────────────────
alpha_map = {'WATCH':0.18, 'DANGER':0.35, 'CRITICAL':0.55}

for _, row in trail_df.iterrows():
    ox, oy, oz = compute_orbit_trail(row)
    cat   = row['category']
    alpha = alpha_map.get(cat, 0.10)
    col   = COLOR_MAP.get(cat, '#00c853')

    fig.add_trace(go.Scatter3d(
        x=ox, y=oy, z=oz,
        mode='lines',
        line=dict(color=col, width=1.2),
        opacity=alpha,
        hoverinfo='skip',
        showlegend=False,
    ))

# ── 8d: 48-Hour Boundary Ring ─────────────────────────────────────────────────
# Represents the operational 2-day cutoff constraint of SCRAP
r_boundary = 1.35   # ~2,230 km altitude
phi_ring   = np.linspace(0, 2*np.pi, 300)
Xr = r_boundary * np.cos(phi_ring)
Yr = r_boundary * np.sin(phi_ring)
Zr = np.zeros(300)

fig.add_trace(go.Scatter3d(
    x=Xr, y=Yr, z=Zr,
    mode='lines',
    line=dict(color='rgba(0,212,255,0.6)', width=3),
    name='48-hr TCA Boundary',
    hovertemplate='<b>48-Hour Operational Boundary</b><br>All predictions made before this ring<extra></extra>',
))

# ── 8e: Debris Scatter Points (by category) ───────────────────────────────────
for cat in ['SAFE', 'WATCH', 'DANGER', 'CRITICAL']:
    sub = df[df['category'] == cat].copy()
    if len(sub) == 0:
        continue

    hover_text = [
        f"<b>{row.event_id}</b><br>"
        f"<b>Category:</b> {row.category}<br>"
        f"<b>P(collision):</b> {row.true_prob:.2e}<br>"
        f"<b>log₁₀(P):</b> {row.true_log_p:.3f}<br>"
        f"<b>Mahal. Distance:</b> {row.mahal_dist:.2f}<br>"
        f"<b>Altitude:</b> {row.alt_km:.0f} km<br>"
        f"<b>TCA:</b> {row.tca_hrs:.1f} hrs<br>"
        f"<b>Sentinel Score:</b> {row.sentinel_score:.4f}<br>"
        f"<b>SCRAP Flagged:</b> {'✅ YES' if row.scrap_flagged else '❌ NO'}<br>"
        f"<b>F10.7 Solar Flux:</b> {row.f107:.1f}"
        for _, row in sub.iterrows()
    ]

    fig.add_trace(go.Scatter3d(
        x=sub['x'], y=sub['y'], z=sub['z'],
        mode='markers',
        name=cat,
        marker=dict(
            size=sub['size'],
            color=COLOR_MAP[cat],
            opacity=OPACITY_MAP[cat],
            symbol='circle',
            line=dict(color='rgba(0,0,0,0.2)', width=0.5) if cat != 'SAFE' else dict(width=0),
        ),
        text=hover_text,
        hovertemplate='%{text}<extra></extra>',
    ))

# ── 8f: False Negatives (highlight in white border) ───────────────────────────
fn_df = df[df['is_fn']]
if len(fn_df) > 0:
    hover_fn = [
        f"<b>⚠️ FALSE NEGATIVE</b><br>"
        f"<b>{row.event_id}</b><br>"
        f"<b>True Risk:</b> {row.true_prob:.2e}<br>"
        f"<b>Sentinel:</b> {row.sentinel_score:.4f} (missed t*={SCRAP['sentinel_threshold']})<br>"
        f"<b>Altitude:</b> {row.alt_km:.0f} km<br>"
        f"This represents SCRAP's irreducible Bayes error"
        for _, row in fn_df.iterrows()
    ]

    fig.add_trace(go.Scatter3d(
        x=fn_df['x'], y=fn_df['y'], z=fn_df['z'],
        mode='markers',
        name=f'False Negative (FN={SCRAP["fn"]})',
        marker=dict(
            size=16,
            color='rgba(0,0,0,0)',
            symbol='circle',
            line=dict(color='white', width=2.5),
        ),
        text=hover_fn,
        hovertemplate='%{text}<extra></extra>',
    ))

# ── 8g: Layout & Style ────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            "<b>SCRAP — 3D Orbital Risk Visualization</b><br>"
            "<sup>XGBoost Specialist · F₂=0.9474 · Recall=97% · Precision=86.6%<br>"
            "162,634 CDMs · 13,154 Events · 48-hr Operational Constraint</sup>"
        ),
        x=0.01, y=0.98,
        font=dict(size=16, color='white', family='monospace'),
    ),
    scene=dict(
        xaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
                   backgroundcolor='rgba(0,0,0,0)', title=''),
        yaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
                   backgroundcolor='rgba(0,0,0,0)', title=''),
        zaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
                   backgroundcolor='rgba(0,0,0,0)', title=''),
        bgcolor='#000408',
        camera=dict(
            eye=dict(x=1.5, y=0.8, z=1.5),
            up=dict(x=0, y=1, z=0),
        ),
        aspectmode='cube',
    ),
    paper_bgcolor='#000408',
    plot_bgcolor='#000408',
    font=dict(color='#8ab4cc', family='monospace'),
    legend=dict(
        x=0.01, y=0.88,
        bgcolor='rgba(0,8,20,0.75)',
        bordercolor='rgba(0,212,255,0.3)',
        borderwidth=1,
        font=dict(size=11, color='#8ab4cc'),
    ),
    margin=dict(l=0, r=0, t=60, b=0),
    height=750,
)

fig.show()


## Risk Distribution Dashboard

In [91]:
fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Risk Distribution (log₁₀ P)',
        'Sentinel Score Distribution',
        'Altitude vs. Risk Level',
        'SCRAP Confusion Matrix',
        'Mahalanobis Distance vs. Risk',
        'Prediction Error (XGBoost RMSE)',
    ],
    specs=[
        [{'type':'xy'}, {'type':'xy'}, {'type':'xy'}],
        [{'type':'xy'}, {'type':'xy'}, {'type':'xy'}],
    ],
    horizontal_spacing=0.1,
    vertical_spacing=0.18,
)

COLORS_ORDER = ['#00c853','#ffd740','#ff6d00','#ff1744']
CATS_ORDER   = ['SAFE','WATCH','DANGER','CRITICAL']

# ── Plot 1: Risk histogram ────────────────────────────────────────────────────
for cat, col in zip(CATS_ORDER, COLORS_ORDER):
    sub = df[df['category']==cat]['true_log_p']
    fig2.add_trace(go.Histogram(
        x=sub, name=cat, marker_color=col, opacity=0.75,
        nbinsx=20, showlegend=True,
    ), row=1, col=1)

fig2.add_vline(x=-6, line_dash='dash', line_color='#00d4ff', line_width=2,
               annotation_text='10⁻⁶ threshold', annotation_font_color='#00d4ff', row=1, col=1)

# ── Plot 2: Sentinel score distribution ──────────────────────────────────────
fig2.add_trace(go.Histogram(
    x=df[~df['is_high_risk']]['sentinel_score'],
    name='Safe events', marker_color='#00c853', opacity=0.6, nbinsx=30, showlegend=False,
), row=1, col=2)
fig2.add_trace(go.Histogram(
    x=df[df['is_high_risk']]['sentinel_score'],
    name='High-risk events', marker_color='#ff1744', opacity=0.8, nbinsx=30, showlegend=False,
), row=1, col=2)
fig2.add_vline(x=SCRAP['sentinel_threshold'], line_dash='dash', line_color='#00d4ff',
               line_width=2, annotation_text=f't*={SCRAP["sentinel_threshold"]}',
               annotation_font_color='#00d4ff', row=1, col=2)

# ── Plot 3: Altitude vs risk level (box) ─────────────────────────────────────
for cat, col in zip(CATS_ORDER, COLORS_ORDER):
    sub = df[df['category']==cat]['alt_km']
    fig2.add_trace(go.Box(
        y=sub, name=cat, marker_color=col, showlegend=False,
        boxmean=True,
    ), row=1, col=3)

# ── Plot 4: Confusion matrix heatmap ─────────────────────────────────────────
cm = np.array([[SCRAP['tn'], SCRAP['fp']], [SCRAP['fn'], SCRAP['tp']]])
cm_text = [[f"TN<br>{SCRAP['tn']}", f"FP<br>{SCRAP['fp']}"],
           [f"FN⚠<br>{SCRAP['fn']}", f"TP✓<br>{SCRAP['tp']}"]]

fig2.add_trace(go.Heatmap(
    z=cm,
    x=['Predicted Safe','Predicted Danger'],
    y=['True Safe','True Danger'],
    text=cm_text,
    texttemplate='%{text}',
    textfont=dict(size=14, color='white'),
    colorscale=[[0,'#0a1628'],[0.5,'#1a4a8a'],[1,'#00c853']],
    showscale=False,
    name='Confusion Matrix',
), row=2, col=1)

# ── Plot 5: Mahalanobis vs risk ───────────────────────────────────────────────
for cat, col in zip(CATS_ORDER, COLORS_ORDER):
    sub = df[df['category']==cat]
    fig2.add_trace(go.Scatter(
        x=sub['mahal_dist'], y=sub['true_log_p'],
        mode='markers',
        marker=dict(color=col, size=4, opacity=0.6),
        name=cat, showlegend=False,
    ), row=2, col=2)

fig2.add_hline(y=-6, line_dash='dash', line_color='#00d4ff', line_width=1.5, row=2, col=2)

# ── Plot 6: Prediction residuals (RMSE) ──────────────────────────────────────
residuals = df['pred_log_p'] - df['true_log_p']
fig2.add_trace(go.Histogram(
    x=residuals[df['is_high_risk']],
    name='High-risk residuals',
    marker_color='#ff6d00', opacity=0.8, nbinsx=25, showlegend=False,
), row=2, col=3)
fig2.add_vline(x=0, line_dash='dash', line_color='white', line_width=1.5, row=2, col=3)
fig2.add_vline(x=SCRAP['rmse'], line_dash='dot', line_color='#00d4ff', line_width=1.5,
               annotation_text=f'+RMSE={SCRAP["rmse"]}', annotation_font_color='#00d4ff', row=2, col=3)
fig2.add_vline(x=-SCRAP['rmse'], line_dash='dot', line_color='#00d4ff', line_width=1.5, row=2, col=3)

# ── Dashboard layout ─────────────────────────────────────────────────────────
fig2.update_layout(
    title=dict(
        text='<b>SCRAP v10 — Model Diagnostics Dashboard</b>',
        x=0.5, font=dict(size=16, color='white', family='monospace'),
    ),
    paper_bgcolor='#000f22',
    plot_bgcolor='#000f22',
    font=dict(color='#8ab4cc', family='monospace', size=10),
    height=680,
    barmode='overlay',
    legend=dict(
        bgcolor='rgba(0,8,20,0.8)',
        bordercolor='rgba(0,212,255,0.3)',
        borderwidth=1,
        font=dict(size=10),
    ),
)
fig2.update_xaxes(gridcolor='rgba(0,50,80,0.4)', zerolinecolor='rgba(0,80,120,0.4)')
fig2.update_yaxes(gridcolor='rgba(0,50,80,0.4)', zerolinecolor='rgba(0,80,120,0.4)')

fig2.show()